# Linear Regression: New York Air Quality

Editable regression blocks for coefficient tests, linear combinations, and general linear F-tests.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))
import pandas as pd
from regression_toolbox import RegressionToolbox

# Dataset choice
# Keep True for the included default dataset.
# Set to False to use your own dataset, then enter the file name WITH extension.
# Put custom files either in the project folder or in the data/ folder.
use_default_dataset = True
custom_file_name = "your_regression_data.csv"  # examples: my_data.csv, my_data.xlsx, my_data.json

def choose_data_file(default_file_name, custom_file_name, use_default_dataset):
    if use_default_dataset:
        return ROOT / 'data' / default_file_name
    if not custom_file_name or custom_file_name == "your_regression_data.csv":
        raise ValueError("Set custom_file_name to your file name with extension, for example 'my_data.csv'.")
    custom_path = Path(custom_file_name)
    if custom_path.suffix.lower() not in {'.csv', '.xlsx', '.xls', '.json'}:
        raise ValueError("Custom file must include a supported extension: .csv, .xlsx, .xls, or .json")
    candidates = [custom_path, ROOT / custom_path, ROOT / 'data' / custom_path.name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {custom_file_name}. Put it in the project folder or data/ folder.")

def validate_for_regression(df, y_col, x_cols):
    required = [y_col] + list(x_cols)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns for regression: {missing}. Available columns: {list(df.columns)}")
    numeric_df = df[required].apply(pd.to_numeric, errors='coerce').dropna()
    if len(numeric_df) <= len(x_cols) + 1:
        raise ValueError("Regression needs more complete numeric rows than the number of coefficients being estimated.")
    for col in required:
        if numeric_df[col].nunique() < 2:
            raise ValueError(f"Column '{col}' must vary; it cannot be constant for regression.")

# If using a custom dataset, update these column names and the later model blocks to match your file.
y_col = 'Ozone'
x_cols = ['Temp', 'Wind', 'Solar.R']

data_path = choose_data_file('airquality.csv', custom_file_name, use_default_dataset)
print('Loaded file:', data_path)
tool = RegressionToolbox(data_path)
df = tool.data
validate_for_regression(df, y_col, x_cols)
display(df.head())
print('Shape:', df.shape)
display(df.isna().sum().to_frame('missing'))
display(tool.summarize_data())

## Block 1: Simple linear regression

In [ ]:
x_col = 'Temp'
y_col = 'Ozone'
alpha = 0.05

simple = tool.fit_simple_linear_regression(x_col, y_col, alpha)
display(tool.coefficient_table(simple))
display(tool.anova_regression_table(simple))
print('R-squared:', tool.calculate_r_squared(simple))
tool.plot_simple_regression(simple)
tool.plot_residuals_vs_fitted(simple)
tool.plot_qq_residuals(simple);

## Block 2: Multiple linear regression

In [ ]:
x_cols = ['Temp', 'Wind', 'Solar.R']
y_col = 'Ozone'
alpha = 0.05

model = tool.fit_multiple_linear_regression(x_cols, y_col, alpha)
display(tool.coefficient_table(model))
display(tool.anova_regression_table(model))
print('Coefficient order:', model['coefficient_names'])
tool.plot_observed_vs_fitted(model)
tool.plot_residuals_vs_fitted(model)
tool.plot_qq_residuals(model);

## Block 3: Coefficient t-test

In [ ]:
coefficient = 'Temp'
null_value = 0
alternative = 'greater'
alpha = 0.05

test = tool.coefficient_t_test(model, coefficient, null_value, alternative, alpha)
display(__import__('pandas').DataFrame({k: [v] for k, v in test.items() if k not in ['coefficient_names', 'contrast_vector']}))
tool.plot_t_distribution_for_coefficient(test);

## Block 4: Linear combination test
Use the coefficient order printed above. Example: `[0, 1, 0, 0]` tests the temperature slope.

In [ ]:
contrast_vector = [0, 1, 0, 0]
null_value = 0
alternative = 'greater'
alpha = 0.05

lin_test = tool.linear_combination_test(model, contrast_vector, null_value, alternative, alpha)
display(__import__('pandas').DataFrame({k: [v] for k, v in lin_test.items() if k not in ['coefficient_names', 'contrast_vector']}))
print('Coefficient order:', lin_test['coefficient_names'])
print('Contrast vector:', lin_test['contrast_vector'])
tool.plot_t_distribution_for_linear_combination(lin_test);

## Block 5: General linear F-test

In [ ]:
C = [[0, 0, 1, 0], [0, 0, 0, 1]]
d = [0, 0]
alpha = 0.05

f_test = tool.general_linear_f_test(model, C, d, alpha, label='Joint test: Wind = Solar.R = 0')
display(__import__('pandas').DataFrame({k: [v] for k, v in f_test.items() if k not in ['C', 'd']}))
tool.plot_f_distribution_for_model(f_test);